# RFM Customer Segmentation Analysis


In [17]:
import pandas as pd
from sqlalchemy import create_engine

# ---
# Why we are doing this:
# We are connecting directly to the DuckDB data warehouse file.
# This allows us to access the clean, tested, and transformed data
# from our dbt pipeline instead of working with raw, messy CSV files.
# This is the standard, professional workflow.
# ---

# Create a connection to the DuckDB database file
# The database file is located in the `dbt_warehouse` directory at the root of the project.
engine = create_engine('duckdb:///../dbt_warehouse/olist.duckdb')

# Write a simple SQL query to select everything from our final RFM model
query = "SELECT * FROM mart_rfm"

# Execute the query and load the results into a pandas DataFrame
rfm_df = pd.read_sql(query, engine)

# Display the first few rows to verify the data was loaded correctly

rfm_df[rfm_df['frequency'] > 5].head(10)



,customer_unique_id,recency,frequency,monetary
320,63cfc61cee11cbe306bff5857d00bfe4,93,6,826.32
8950,3e43e6105506432c953e165fb2acf44c,183,9,1172.66
9619,ca77025e7201e3b30c44b472ff346268,89,7,1122.72
26933,f0e310a6839dce9de1638e0fe5ab282a,146,6,540.69
38228,47c1a3033b8b77b3ab6e109eb4d5fdf3,217,6,944.21
55880,dc813062e0fc23409cd255f7f53c7074,6,6,1094.63
61683,1b6c7548a2a1f9037c1fd3ddfed95f33,196,7,959.01
67789,8d50f5eadf50201ccdcedfb9e2ac8455,9,15,879.27
76541,6469f99c1f9dfae7733b25662e7f1782,62,7,758.83
85584,12f5d6e1cbf93dafd9dcc19095df0b3d,601,6,110.72


## Calculate RFM Scores

Now that we have the raw RFM values, we need to score each customer on a scale of 1-5 for each metric. This allows us to easily group and compare customers. We will use quintiles (dividing the data into 5 equal parts) to create these scores.


In [5]:
# ---
# Why we are doing this:
# Raw R, F, and M values are on different scales. Scoring them from 1-5
# standardizes them, making it easy to compare and combine them.
# `pd.qcut` is used for R and M, but F (frequency) is often heavily skewed
# (many customers with 1 purchase), so we use a custom function for it.
# ---

# Create labels for our scores (1 is worst, 5 is best)
r_labels = range(5, 0, -1) # For Recency, lower is better, so we reverse the labels
fm_labels = range(1, 6)    # For Frequency and Monetary, higher is better

# Calculate R and M scores using quintiles
rfm_df['R_score'] = pd.qcut(rfm_df['recency'], q=5, labels=r_labels, duplicates='drop').astype(int)
rfm_df['M_score'] = pd.qcut(rfm_df['monetary'], q=5, labels=fm_labels, duplicates='drop').astype(int)

# Define a function to score frequency based on common business rules
def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x == 4:
        return 4
    else: # 5 or more purchases
        return 5

# Apply the function to the frequency column
rfm_df['F_score'] = rfm_df['frequency'].apply(frequency_score)

# ---
# Why we are doing this:
# The combined RFM score gives us a single, comparable metric for each customer.
# We treat the scores as strings and concatenate them (e.g., a customer with 5 for R, 1 for F, and 3 for M becomes '513').
# ---

# Combine the individual scores into a single RFM_Score
def join_rfm(x):
    return str(x['R_score']) + str(x['F_score']) + str(x['M_score'])

rfm_df['RFM_Score'] = rfm_df.apply(join_rfm, axis=1)

# Display the first few rows with the new scores
rfm_df.head()


,customer_unique_id,recency,frequency,monetary,R_score,M_score,F_score,RFM_Score
0,addec96d2e059c80c30fe6871d30d177,191,1,22.77,3,1,1,311
1,66cc90195ca44cc7ac6a1cd0e1e1e7b2,324,1,30.40,2,1,1,211
2,8d46223c91cbeb93e0930ca8bd8ffca2,276,1,171.32,2,4,1,214
3,27cf4b153010911a0957150255a6c6db,137,1,465.40,4,5,1,415
4,be1e99a0c57d7c3c699cfc4db26c8edf,29,1,40.27,5,1,1,511


### Analyzing the Frequency Distribution

Before settling on our scoring logic, let's analyze the distribution of the `frequency` column. This will help us understand why `pd.qcut` failed and validate if our static business rules are reasonable. This is a key "Lab" step to inform our "Factory" logic.


In [ ]:
# ---
# Why we are doing this:
# We need to understand the shape of our data to make good modeling choices.
# `value_counts` will show us exactly how many customers made 1, 2, 3, etc. purchases.
# `describe` will give us the statistical summary, including the percentiles.
# ---

# Show the count of customers for each frequency value
print("Frequency Value Counts:")
print(rfm_df['frequency'].value_counts(normalize=True).head())
print("\\n-------------------------\\n")

# Show the statistical summary of the frequency
print("Frequency Statistical Summary:")
print(rfm_df['frequency'].describe())

